### Kernel: R

## Avaliação dos Modelos

In [1]:
# Configuração inicial
rm(list = ls(all = TRUE))

library(pacman)
p_load(tidyverse, survival, smcure, readr, tidyr, dplyr)

seed <- 123
set.seed(seed)


options(scipen = 999)

In [2]:
# Leitura dos dados
outcomes_teste <- read_csv("../dados/dados-processados/dados_teste.csv", show_col_types = FALSE)

# Leitura dos arquivos de saida

pred_cox <- read_csv("../ajuste-modelos/cox-cura/dados/saida/predicao_cox.csv", show_col_types = FALSE)
pred_cura <- read_csv("../ajuste-modelos/cox-cura/dados/saida/predicao_cura.csv", show_col_types = FALSE)
pred_deepsurv_covsel <- read_csv("../ajuste-modelos/deepsurv/dados/saida/predicao-deepsurv-covariaveis-selecionadas.csv", show_col_types = FALSE)
pred_rede_A_covsel <- read_csv("../ajuste-modelos/pseudo-sobrevivencia-total/dados/saida/predicao-rede-A-onehot-covsel.csv", show_col_types = FALSE)
pred_rede_B_covsel <- read_csv("../ajuste-modelos/pseudo-sobrevivencia-total/dados/saida/predicao-rede-B-tempo-continuo-covsel.csv", show_col_types = FALSE)

# Experimentos complementares
pred_rede_A <- read_csv("../experimentos-complementares/pseudo/sobrevivencia-total/dados/saida/predicao-rede-A-onehot.csv", show_col_types = FALSE)
pred_rede_B <- read_csv("../experimentos-complementares/pseudo/sobrevivencia-total/dados/saida/predicao-rede-B-tempo-continuo.csv", show_col_types = FALSE)
pred_rede_C <- read_csv("../experimentos-complementares/pseudo/sobrevivencia-total/dados/saida/predicao-rede-C-optuna.csv", show_col_types = FALSE)
pred_deepsurv <- read_csv("../experimentos-complementares/deepsurv/dados/saida/predicao-deepsurv.csv", show_col_types = FALSE)
pred_deepsurv_optuna <- read_csv("../experimentos-complementares/deepsurv/dados/saida/predicao-deepsurv-optuna.csv", show_col_types = FALSE)
pred_deepsurv_optuna_covsel <- read_csv("../experimentos-complementares/deepsurv/dados/saida/predicao-deepsurv-optuna-covariaveis-selecionadas.csv", show_col_types = FALSE)
pred_rede_lat_inc <- read_csv("../experimentos-complementares/pseudo/latencia-incidencia/dados/saida/predicao-rede-lat-inc.csv", show_col_types = FALSE)
pred_rede_lat_inc_covsel <- read_csv("../experimentos-complementares/pseudo/latencia-incidencia/dados/saida/predicao-rede-lat-inc-covsel.csv", show_col_types = FALSE)

## Avaliação Time Dependent Concordance Index

In [3]:
calc_ctd <- function(outcomes_df, pred_event_df) {
  outcomes <- outcomes_df %>%
    transmute(
      id_paciente = as.numeric(id_paciente),
      tempo = as.numeric(tempo),
      delta_t = as.integer(delta_t)
    ) %>%
    distinct(id_paciente, .keep_all = TRUE) %>%
    arrange(tempo)

  n_pacientes <- nrow(outcomes)

  pred_mat_df <- pred_event_df %>%
    transmute(
      ID = as.numeric(ID),
      TIME = as.numeric(TIME),
      PRED_S = as.numeric(PRED_S)
    ) %>%
    distinct(ID, TIME, .keep_all = TRUE) %>%
    pivot_wider(names_from = TIME, values_from = PRED_S) %>%
    arrange(match(ID, outcomes$id_paciente))

# Validação de correspondência entre pacientes e tempos
  stopifnot(all(pred_mat_df$ID %in% outcomes$id_paciente)) 
  stopifnot(all(outcomes$id_paciente %in% pred_mat_df$ID))   
  stopifnot(nrow(pred_mat_df) == nrow(outcomes))

  matriz_pred <- as.matrix(pred_mat_df %>% select(-ID))
  time_vals <- as.numeric(colnames(matriz_pred))



  concordante <- 0
  empates <- 0
  pares_comparaveis <- 0

  tempos_out <- outcomes$tempo
  delta_out <- outcomes$delta_t

  for (i in seq_len(n_pacientes)) {
    if (delta_out[i] != 1) next

    t_i <- tempos_out[i]
    idx_col_t <- which(time_vals == t_i)[1]
    if (is.na(idx_col_t)) next

    risk_i <- 1 - matriz_pred[i, idx_col_t]
    if (is.na(risk_i)) next

    idx_j <- which(tempos_out > t_i)
    if (length(idx_j) == 0) next

    risks_j <- 1 - matriz_pred[idx_j, idx_col_t]
    validos <- !is.na(risks_j)
    risks_j_validos <- risks_j[validos]
    if (length(risks_j_validos) == 0) next

    pares_comparaveis <- pares_comparaveis + length(risks_j_validos)
    concordante <- concordante + sum(risk_i > risks_j_validos)
    empates <- empates + sum(risk_i == risks_j_validos)
  }

  ctd <- if (pares_comparaveis > 0) {
    (concordante + 0.5 * empates) / pares_comparaveis
  } else {
    NA_real_
  }

  c(
    CTD_TESTE = ctd,
    COMPARAVEIS = pares_comparaveis,
    CONCORDANTES = concordante,
    EMPATES = empates
  )
}

In [4]:
# Avaliação dos modelos usando o CTD

met_cox <- calc_ctd(outcomes_teste, pred_cox)
met_cura <- calc_ctd(outcomes_teste, pred_cura)
met_rede_A <- calc_ctd(outcomes_teste, pred_rede_A)
met_rede_A_covsel <- calc_ctd(outcomes_teste, pred_rede_A_covsel)
met_rede_B <- calc_ctd(outcomes_teste, pred_rede_B)
met_rede_B_covsel <- calc_ctd(outcomes_teste, pred_rede_B_covsel)
met_rede_C <- calc_ctd(outcomes_teste, pred_rede_C)
met_deepsurv <- calc_ctd(outcomes_teste, pred_deepsurv)
met_deepsurv_covsel <- calc_ctd(outcomes_teste, pred_deepsurv_covsel)
met_deepsurv_optuna <- calc_ctd(outcomes_teste, pred_deepsurv_optuna)
met_deepsurv_optuna_covsel <- calc_ctd(outcomes_teste, pred_deepsurv_optuna_covsel)
met_lat_inc <- calc_ctd(outcomes_teste, pred_rede_lat_inc)
met_lat_inc_covsel <- calc_ctd(outcomes_teste, pred_rede_lat_inc_covsel)

avaliacao_ctd <- tibble::tibble(
  MODELO = c(
    "cox",
    "cura",
    "rede_A_onehot",
    "rede_A_onehot_covsel",
    "rede_B_continuo",
    "rede_B_continuo_covsel",
    "rede_C_optuna",
    "deepsurv",
    "deepsurv_covsel",
    "deepsurv_optuna",
    "deepsurv_optuna_covsel",
    "rede_lat_inc",
    "rede_lat_inc_covsel"
  ),
  CTD_TESTE = c(
    as.numeric(met_cox["CTD_TESTE"]),
    as.numeric(met_cura["CTD_TESTE"]),
    as.numeric(met_rede_A["CTD_TESTE"]),
    as.numeric(met_rede_A_covsel["CTD_TESTE"]),
    as.numeric(met_rede_B["CTD_TESTE"]),
    as.numeric(met_rede_B_covsel["CTD_TESTE"]),
    as.numeric(met_rede_C["CTD_TESTE"]),
    as.numeric(met_deepsurv["CTD_TESTE"]),
    as.numeric(met_deepsurv_covsel["CTD_TESTE"]),
    as.numeric(met_deepsurv_optuna["CTD_TESTE"]),
    as.numeric(met_deepsurv_optuna_covsel["CTD_TESTE"]),
    as.numeric(met_lat_inc["CTD_TESTE"]),
    as.numeric(met_lat_inc_covsel["CTD_TESTE"])
  ),
  COMPARAVEIS = c(
    as.numeric(met_cox["COMPARAVEIS"]),
    as.numeric(met_cura["COMPARAVEIS"]),
    as.numeric(met_rede_A["COMPARAVEIS"]),
    as.numeric(met_rede_A_covsel["COMPARAVEIS"]),
    as.numeric(met_rede_B["COMPARAVEIS"]),
    as.numeric(met_rede_B_covsel["COMPARAVEIS"]),
    as.numeric(met_rede_C["COMPARAVEIS"]),
    as.numeric(met_deepsurv["COMPARAVEIS"]),
    as.numeric(met_deepsurv_covsel["COMPARAVEIS"]),
    as.numeric(met_deepsurv_optuna["COMPARAVEIS"]),
    as.numeric(met_deepsurv_optuna_covsel["COMPARAVEIS"]),
    as.numeric(met_lat_inc["COMPARAVEIS"]),
    as.numeric(met_lat_inc_covsel["COMPARAVEIS"])
  ),
  CONCORDANTES = c(
    as.numeric(met_cox["CONCORDANTES"]),
    as.numeric(met_cura["CONCORDANTES"]),
    as.numeric(met_rede_A["CONCORDANTES"]),
    as.numeric(met_rede_A_covsel["CONCORDANTES"]),
    as.numeric(met_rede_B["CONCORDANTES"]),
    as.numeric(met_rede_B_covsel["CONCORDANTES"]),
    as.numeric(met_rede_C["CONCORDANTES"]),
    as.numeric(met_deepsurv["CONCORDANTES"]),
    as.numeric(met_deepsurv_covsel["CONCORDANTES"]),
    as.numeric(met_deepsurv_optuna["CONCORDANTES"]),
    as.numeric(met_deepsurv_optuna_covsel["CONCORDANTES"]),
    as.numeric(met_lat_inc["CONCORDANTES"]),
    as.numeric(met_lat_inc_covsel["CONCORDANTES"])
  ),
  EMPATES = c(
    as.numeric(met_cox["EMPATES"]),
    as.numeric(met_cura["EMPATES"]),
    as.numeric(met_rede_A["EMPATES"]),
    as.numeric(met_rede_A_covsel["EMPATES"]),
    as.numeric(met_rede_B["EMPATES"]),
    as.numeric(met_rede_B_covsel["EMPATES"]),
    as.numeric(met_rede_C["EMPATES"]),
    as.numeric(met_deepsurv["EMPATES"]),
    as.numeric(met_deepsurv_covsel["EMPATES"]),
    as.numeric(met_deepsurv_optuna["EMPATES"]),
    as.numeric(met_deepsurv_optuna_covsel["EMPATES"]),
    as.numeric(met_lat_inc["EMPATES"]),
    as.numeric(met_lat_inc_covsel["EMPATES"])
  )
)

print(avaliacao_ctd)

# A tibble: 13 × 5
   MODELO                 CTD_TESTE COMPARAVEIS CONCORDANTES EMPATES
   <chr>                      <dbl>       <dbl>        <dbl>   <dbl>
 1 cox                        0.683      129315        83080   10432
 2 cura                       0.701      129315        87234    6847
 3 rede_A_onehot              0.633      129315        63738   36359
 4 rede_A_onehot_covsel       0.695      129315        86439    6847
 5 rede_B_continuo            0.576      129315        59283   30410
 6 rede_B_continuo_covsel     0.692      129315        86080    6847
 7 rede_C_optuna              0.596      129315        70630   12892
 8 deepsurv                   0.655      129315        84737       0
 9 deepsurv_covsel            0.690      129315        85845    6847
10 deepsurv_optuna            0.663      129315        85724       0
11 deepsurv_optuna_covsel     0.693      129315        86189    6839
12 rede_lat_inc               0.634      129315        81922      14
13 rede_lat_inc

## Avaliação Integrated Breir Score

In [5]:
calc_ibs <- function(outcomes_df, pred_event_df) {
  df <- outcomes_df %>%
    transmute(
      id_paciente = as.numeric(id_paciente),
      tempo = as.numeric(tempo),
      delta_t = as.integer(delta_t)
    ) %>%
    distinct(id_paciente, .keep_all = TRUE)

  n_total <- nrow(df)

  pred_mat <- pred_event_df %>%
    transmute(ID = as.numeric(ID), TIME = as.numeric(TIME), PRED_S = as.numeric(PRED_S)) %>%
    distinct(ID, TIME, .keep_all = TRUE) %>%
    pivot_wider(names_from = TIME, values_from = PRED_S) %>%
    arrange(match(ID, df$id_paciente))

  matriz_pred <- as.matrix(pred_mat %>% select(-ID))
  grade_ibs <- as.numeric(colnames(matriz_pred))

  km_censura <- survival::survfit(survival::Surv(tempo, 1 - delta_t) ~ 1, data = df)
  obter_G_vetorizado <- stepfun(km_censura$time, pmax(c(1, km_censura$surv), 0.001))
  G_t_i <- obter_G_vetorizado(df$tempo)

  bs_valores <- numeric(length(grade_ibs))

  for (k in seq_along(grade_ibs)) {
    t_eval <- grade_ibs[k]
    g_t_eval <- obter_G_vetorizado(t_eval)
    s_pred <- matriz_pred[, k]
    cond_1 <- (df$tempo <= t_eval) & (df$delta_t == 1)
    cond_2 <- (df$tempo > t_eval)
    peso_1 <- ifelse(cond_1, 1 / G_t_i, 0)
    peso_2 <- ifelse(cond_2, 1 / g_t_eval, 0)
    erro <- peso_1 * (0 - s_pred)^2 + peso_2 * (1 - s_pred)^2
    erro[is.na(erro)] <- 0
    bs_valores[k] <- sum(erro) / n_total
  }

  delta_t <- grade_ibs[2] - grade_ibs[1]
  integral_bs <- sum(bs_valores) * delta_t
  ibs_final <- integral_bs / (max(grade_ibs) - min(grade_ibs))

  list(IBS = ibs_final, BS = bs_valores, grade = grade_ibs)
}

In [6]:
# IBS para os modelos

ibs_cox <- calc_ibs(outcomes_teste, pred_cox)
ibs_cura <- calc_ibs(outcomes_teste, pred_cura)
ibs_rede_A <- calc_ibs(outcomes_teste, pred_rede_A)
ibs_rede_A_covsel <- calc_ibs(outcomes_teste, pred_rede_A_covsel)
ibs_rede_B <- calc_ibs(outcomes_teste, pred_rede_B)
ibs_rede_B_covsel <- calc_ibs(outcomes_teste, pred_rede_B_covsel)
ibs_rede_C <- calc_ibs(outcomes_teste, pred_rede_C)
ibs_deepsurv <- calc_ibs(outcomes_teste, pred_deepsurv)
ibs_deepsurv_covsel <- calc_ibs(outcomes_teste, pred_deepsurv_covsel)
ibs_deepsurv_optuna <- calc_ibs(outcomes_teste, pred_deepsurv_optuna)
ibs_deepsurv_optuna_covsel <- calc_ibs(outcomes_teste, pred_deepsurv_optuna_covsel)
ibs_lat_inc <- calc_ibs(outcomes_teste, pred_rede_lat_inc)
ibs_lat_inc_covsel <- calc_ibs(outcomes_teste, pred_rede_lat_inc_covsel)

avaliacao_IBS <- tibble(
  MODELO = c(
    "cox",
    "cura",
    "rede_A_onehot",
    "rede_A_onehot_covsel",
    "rede_B_continuo",
    "rede_B_continuo_covsel",
    "rede_C_optuna",
    "deepsurv",
    "deepsurv_covsel",
    "deepsurv_optuna",
    "deepsurv_optuna_covsel",
    "rede_lat_inc",
    "rede_lat_inc_covsel"
  ),
  IBS = c(
    as.numeric(ibs_cox$IBS),
    as.numeric(ibs_cura$IBS),
    as.numeric(ibs_rede_A$IBS),
    as.numeric(ibs_rede_A_covsel$IBS),
    as.numeric(ibs_rede_B$IBS),
    as.numeric(ibs_rede_B_covsel$IBS),
    as.numeric(ibs_rede_C$IBS),
    as.numeric(ibs_deepsurv$IBS),
    as.numeric(ibs_deepsurv_covsel$IBS),
    as.numeric(ibs_deepsurv_optuna$IBS),
    as.numeric(ibs_deepsurv_optuna_covsel$IBS),
    as.numeric(ibs_lat_inc$IBS),
    as.numeric(ibs_lat_inc_covsel$IBS)
  )
)

print(avaliacao_IBS)

# A tibble: 13 × 2
   MODELO                    IBS
   <chr>                   <dbl>
 1 cox                    0.0949
 2 cura                   0.0946
 3 rede_A_onehot          0.141 
 4 rede_A_onehot_covsel   0.0948
 5 rede_B_continuo        0.130 
 6 rede_B_continuo_covsel 0.0960
 7 rede_C_optuna          0.133 
 8 deepsurv               0.102 
 9 deepsurv_covsel        0.0953
10 deepsurv_optuna        0.0964
11 deepsurv_optuna_covsel 0.0952
12 rede_lat_inc           0.133 
13 rede_lat_inc_covsel    0.0954


## Resultado completo dos modelos

In [7]:
# Resultados ordenados por CTD e IBS

avaliacao_final <- avaliacao_ctd %>%
  left_join(avaliacao_IBS, by = "MODELO") %>%
  select(MODELO, IBS, CTD_TESTE, COMPARAVEIS, CONCORDANTES, EMPATES) %>%
  arrange(desc(CTD_TESTE))

avaliacao_final

MODELO,IBS,CTD_TESTE,COMPARAVEIS,CONCORDANTES,EMPATES
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
cura,0.09457634,0.7010594,129315,87234,6847
rede_A_onehot_covsel,0.09484865,0.6949116,129315,86439,6847
deepsurv_optuna_covsel,0.09518954,0.6929475,129315,86189,6839
rede_B_continuo_covsel,0.09602095,0.6921355,129315,86080,6847
deepsurv_covsel,0.09529531,0.6903182,129315,85845,6847
cox,0.09491912,0.6827978,129315,83080,10432
rede_lat_inc_covsel,0.09537342,0.6728338,129315,83586,6843
deepsurv_optuna,0.09636963,0.6629084,129315,85724,0
deepsurv,0.10224209,0.6552759,129315,84737,0
